# Pympact development
## How many asteroids are out there?

In [3]:
import multineas as neas
from multineas import pympact as pm
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os
from time import time
import spiceypy as spy

data_dir = f"{pm.data_dir}/"
get_ipython = pm.get_ipython()
%load_ext autoreload
%autoreload 2

## Get information about object

In [4]:
# ################################################# #
# Inputs
# ################################################# #
obj_id = pm.check_opts('obj_id','2024YR4')
#orbit_id = pm.check_opts('orbit_id','78')
orbit_id = pm.check_opts('orbit_id','57')
enc_id = pm.check_opts('enc_id','0')

# ################################################# #
# Read orbit
# ################################################# #
object_dir = f"{pm.objects_dir}/{obj_id}"
repo_dir = f"{object_dir}/sim/"
obj_suf = f"obj_{obj_id}-orbit_{orbit_id}"
orbit_file = f"{object_dir}/orbit-{obj_suf}.json"
orbit = pm.load_json(orbit_file)

# ################################################# #
# Key suffixes
# ################################################# #
jd_suf = orbit['encounters'][int(enc_id)]['jd_suf']
jd0_suf = orbit['jd0_suf']
enc_suf = f"{obj_suf}-enc_{jd_suf}"

Loading json data from data/objects/2024YR4/orbit-obj_2024YR4-orbit_57.json


## Generate sample

In [5]:
Nast = int(pm.check_opts('Nast',1000))
print(f"Generating initial conditions for {Nast} asteroids")

# Name of sample
name = pm.check_opts('name','generic')

####################################################
# Load orbit
####################################################
print(f"Loading orbital elements from {orbit['orbit_file']}")
mus,covmat,covlabel = pm.get_mus_cov(orbit)

####################################################
# Orbital elements sample
####################################################
print(f"Generating {Nast} random elements...")
sample_elements = np.random.multivariate_normal(mus,covmat,Nast)

sample_file = f"{repo_dir}/sample-name_{name}-Nast_{Nast}-{orbit['obj_suf']}.csv"
np.savetxt(sample_file,sample_elements)
print(f"\tSaved in {sample_file}")

Generating initial conditions for 1000 asteroids
Loading orbital elements from assets/data/objects/2024_YR4/orbit-obj_2024_YR4-orbit_57.json
Generating 1000 random elements...
	Saved in data/objects/2024YR4/sim//sample-name_generic-Nast_1000-obj_2024_YR4-orbit_57.csv


Read sample:

In [6]:
import rebound as rb
import os   

####################################################
# Solar system initial conditions
####################################################
filesim = f"{repo_dir}/ss-{jd0_suf}.bin"
sim = rb.Simulation()
sim.units = 'kg', 'm', 's'
print(f"Loading solar system initial conditions for jd = {orbit['jd_0']}:")
if not os.path.exists(filesim):
    print(f"\tLoading using Horizons...")
    todos_los_cuerpos= ['Sun','199','299','399','301','499','599','699','799','899']
    sim.add(todos_los_cuerpos, date=f"JD{orbit['jd_0']}")
    sim.move_to_hel()
    sim.save_to_file(filesim)
else:
    print(f"\tLoading from file {filesim}")
    sim = rb.Simulation(filesim)

Loading solar system initial conditions for jd = 2460800.5:
	Loading from file data/objects/2024YR4/sim//ss-jd0_246080050.bin


In [7]:
####################################################
# Orbital elements sample
####################################################
print(f"Reading sample elements...")
sample_elements = pm.np.loadtxt(sample_file)
Nast = len(sample_elements)

####################################################
# Add conditions to simulation
####################################################
for n, elements in enumerate(sample_elements):
    e = elements[0]
    q = elements[1] * neas.constants.au
    jd_p = elements[2]
    node = elements[3] * neas.constants.deg
    peri = elements[4] * neas.constants.deg
    i = elements[5] * neas.constants.deg
    a = q / (1 - e)
    n = pm.np.sqrt(neas.constants.mu_sun / a**3)
    M_0 = pm.np.mod(n * (orbit['jd_0'] - jd_p) * neas.constants.day, 2 * pm.np.pi)
    X_ast_0 = pm.spy.conics([q, e, i, node, peri, M_0, 0, neas.constants.mu_sun], 0)
    r_ast_0 = X_ast_0[:3]
    v_ast_0 = X_ast_0[3:]
    sim.add(m=0,
            x=r_ast_0[0], y=r_ast_0[1], z=r_ast_0[2],
            vx=v_ast_0[0], vy=v_ast_0[1], vz=v_ast_0[2])

####################################################
# Saving simulation state
#################################################
sim.t = orbit['encounters'][int(enc_id)]['time_seg']

sim_suf = f"{jd0_suf}--enc_{jd_suf}"
filesim = f"{repo_dir}/sim-{sim_suf}.bin"
sim.save_to_file(filesim)
print(f"Simulation state saved in {filesim}")

Reading sample elements...
Simulation state saved in data/objects/2024YR4/sim//sim-jd0_246080050--enc_246358909.bin


## Integrate orbits

In [8]:
# Number of time steps
Nt = int(pm.check_opts('Nt',100)) # Number of time steps

# Minimum distance of approach
rplanet = float(pm.check_opts('Rp',1.2))*neas.constants.rearth

In [9]:
####################################################
# Simulation parameters
####################################################
sim_integrator = 'ias15'
sim_dt = 100
sim_collision = 'line'
mus,covmat,covlabel = pm.get_mus_cov(orbit)

In [10]:
####################################################
# Load simulation
####################################################
sim = rb.Simulation(filesim)
tmax = sim.t
sim.t = 0

In [12]:
####################################################
# Prepare simulation properties
####################################################
# Define funciones útiles
Nstart = sim.N
it = 0
start = time()
step = time()
collisions = []

def heartbeat(simulation):
  global it,Nt,start,step,tiempo,tmax,Nstart
  sim = simulation.contents
  if ((sim.t - it*tmax/Nt)>0) or it == 0:
    end = time()
    print(f"Time: {sim.t/tmax:.2e} (it = {it}/{Nt}, Npart = {sim.N}/{Nstart}, elapsed = {end-step:.2f} s, wait = {(Nt-1-it)*(end-step):.2f} s)")
    step = time()
    it += 1

def collision_resolve(simulation,c):
  global collisions,rplanet
  sim = simulation.contents
  sim.move_to_hel()
  xyz_1 = np.array(sim.particles[int(c.p1)].xyz)
  xyz_2 = np.array(sim.particles[int(c.p2)].xyz)
  d = np.linalg.norm(xyz_1-xyz_2)
  collisions += [
      dict(
          t = sim.t,
          p1 = int(c.p1),
          p2 = int(c.p2),
          x1 = xyz_1[0],y1 = xyz_1[1],z1 = xyz_1[2],vx1 = sim.particles[int(c.p1)].vx,vy1 = sim.particles[int(c.p1)].vy,vz1 = sim.particles[int(c.p1)].vz,
          x2 = xyz_2[0],y2 = xyz_2[1],z2 = xyz_2[2],vx2 = sim.particles[int(c.p2)].vx,vy2 = sim.particles[int(c.p2)].vy,vz2 = sim.particles[int(c.p2)].vz,
          d = d/rplanet,
      )
  ]
  return 2

# Modifica simulation
Np = 10
sim.particles[3].r = rplanet
sim.integrator = sim_integrator
sim.dt = sim_dt
sim.collision = sim_collision
sim.collision_resolve = collision_resolve
sim.heartbeat = heartbeat

####################################################
# Integrate
####################################################
Nini = sim.N-Np
print(f"Starting with {Nini} asteroids")

# Integra
tini = time()
sim.integrate(tmax)
end = time()
sim.move_to_hel()
Nend = sim.N-Np
Nimp = (Nini-Nend)
pimp = Nimp/Nini
print(f"Surviving asteroids: {Nend}")
print(f"Impacting asteroids: {Nimp}")
print(f"Impact probability: p = {100*pimp:.2f}%")
print(f"Total execution time: {end-tini:.2f} s = {(end-tini)/60:.2f} min = {(end-tini)/3600:.2f} h")

####################################################
# Save bodies
####################################################
bodies = []
for n in range(sim.N):
    bodies += [
        dict(
            m=sim.particles[n].m,
            x=sim.particles[n].x,
            y=sim.particles[n].y,
            z=sim.particles[n].z,
            vx=sim.particles[n].vx,
            vy=sim.particles[n].vy,
            vz=sim.particles[n].vz,
        )
    ]
bodies = pd.DataFrame(bodies)

Starting with 1000 asteroids
Time: 0.00e+00 (it = 0/100, Npart = 1010/1010, elapsed = 0.02 s, wait = 1.76 s)
Time: 1.04e-02 (it = 1/100, Npart = 1010/1010, elapsed = 1.67 s, wait = 163.60 s)
Time: 2.02e-02 (it = 2/100, Npart = 1010/1010, elapsed = 1.33 s, wait = 129.11 s)
Time: 3.01e-02 (it = 3/100, Npart = 1010/1010, elapsed = 1.40 s, wait = 134.63 s)
Time: 4.03e-02 (it = 4/100, Npart = 1010/1010, elapsed = 1.16 s, wait = 110.30 s)
Time: 5.02e-02 (it = 5/100, Npart = 1010/1010, elapsed = 1.11 s, wait = 104.39 s)
Time: 6.00e-02 (it = 6/100, Npart = 1010/1010, elapsed = 1.13 s, wait = 105.47 s)
Time: 7.03e-02 (it = 7/100, Npart = 1010/1010, elapsed = 1.15 s, wait = 105.52 s)
Time: 8.01e-02 (it = 8/100, Npart = 1010/1010, elapsed = 1.10 s, wait = 99.98 s)
Time: 9.05e-02 (it = 9/100, Npart = 1010/1010, elapsed = 1.15 s, wait = 103.44 s)
Time: 1.00e-01 (it = 10/100, Npart = 1010/1010, elapsed = 1.08 s, wait = 95.81 s)
Time: 1.10e-01 (it = 11/100, Npart = 1010/1010, elapsed = 1.09 s, wait =

In [13]:
# Guarda los resultados
bodies_file = f"{repo_dir}/bodies-{sim_suf}.csv"
print(f"Final conditions of simulation saved to {bodies_file}...")
bodies.to_csv(bodies_file,index=False)

####################################################
# Save candidates including their elements
####################################################
samples = np.loadtxt(sample_file)

# Complete collision data
for c in collisions:
    p2 = c['p2']
    sample = samples[p2-10] # "-10" because the first 10 particles are the planets
    for i,label in enumerate(covlabel):
       c[label] = sample[i]

candidates_file = f"{repo_dir}/candidates-{sim_suf}.csv"
print(f"Collision candidates saved to {candidates_file}...")
collisions = pd.DataFrame(collisions)
collisions.to_csv(candidates_file, index=False)


Final conditions of simulation saved to data/objects/2024YR4/sim//bodies-jd0_246080050--enc_246358909.csv...
Collision candidates saved to data/objects/2024YR4/sim//candidates-jd0_246080050--enc_246358909.csv...


## Complete orbit towards the surface

In [15]:
pm.data_dir

'data'

In [18]:
####################################################
# SPICE kernels
####################################################
spy.furnsh(neas.Util.get_data("kernels/kernels.txt"))

In [19]:
####################################################
# Continue trajectories and impact
####################################################
samples = np.loadtxt(sample_file)
bodies = pd.read_csv(bodies_file)

In [20]:
print(f"Reading candidates from {candidates_file}")
candidates = pd.read_csv(candidates_file)

Ncandidates = len(candidates)
print(f"Number of candidates: {Ncandidates}")

Reading candidates from data/objects/2024YR4/sim//candidates-jd0_246080050--enc_246358909.csv
Number of candidates: 35


In [22]:
omega_earth = 2 * np.pi / 86164.0905
X1_ecliptic = np.array(candidates[['x1', 'y1', 'z1', 'vx1', 'vy1', 'vz1']], dtype=float)
X2_ecliptic = np.array(candidates[['x2', 'y2', 'z2', 'vx2', 'vy2', 'vz2']], dtype=float)
X_relative = X2_ecliptic - X1_ecliptic
timps = np.array(candidates['t']) / neas.constants.day + orbit['jd_0']

mu_earth = neas.constants.G * bodies.iloc[3, 0]
R_earth = neas.constants.rearth

t_abs_imps = np.array(candidates['t'])
impacts = []
ini_index = candidates.columns.get_loc('e')
for i, X in enumerate(X_relative):
    elements = candidates.iloc[i,ini_index:].values
    t_rel_impact, X_impact, X_solution = pm.impact_on_planet(X, mu_earth, R_earth)
    if t_rel_impact is not None:
        impacts += [[i, t_abs_imps[i] + float(t_rel_impact)] + list(X_impact) + list(elements)]
impacts = np.array(impacts)

# Save impacts
Nimpacts = len(impacts)
print(f"Number of impacts: {Nimpacts}")

file_impacts = f"{repo_dir}/impacts-{sim_suf}.csv"
if Nimpacts == 0:
    print(f"No impacts were produced. Try to increase number of initial orbits.")
    print(f"Creating empty impact file {file_impacts}")
    os.system(f"echo > {file_impacts}")
    exit(0)
else: 
    ####################################################
    # Compute conditions on surface
    ####################################################
    print("Computing impact conditions on surface (lon, lat, time)")
    timps = impacts[:, 1] / neas.constants.day + orbit['jd_0']
    locations = []
    for jd, X_ecl in zip(timps, impacts):
        r_ecl = X_ecl[2:5]
        v_ecl = X_ecl[5:8]

        # Time
        date = pm.Time(jd, format='jd', scale='tdb')
        et = pm.spy.unitim(jd, 'JED', 'ET')

        # Get the transformation matrix from ecliptic J2000 to IAU_EARTH
        Meclip2iau = pm.spy.pxform('ECLIPJ2000', 'IAU_EARTH', et)

        # Transform the vector to geodetic cartesian coordinates
        r_earth = pm.spy.mxv(Meclip2iau, np.array(r_ecl))

        # Transform the vector to geodetic latitud and longitud
        lon, lat, h = pm.spy.recgeo(r_earth, neas.constants.rearth, neas.constants.fe)

        # Velocity with respect to the Earth
        v_surface = v_ecl - np.cross(np.array([0, 0, omega_earth]), r_earth)
        v_speed = np.linalg.norm(v_surface)
        z_impact = np.arccos(np.dot(v_surface, -r_earth) / v_speed / np.linalg.norm(r_earth)) * neas.constants.rad

        # Save impact conditions
        location = dict(
            date=date.iso,
            jd=jd,
            lon=lon * neas.constants.rad, lat=lat * neas.constants.rad,
            h=h / 1e3,
            x_ecl=r_ecl[0], y_ecl=r_ecl[1], z_ecl=r_ecl[2],
            vx_ecl=v_ecl[0], vy_ecl=v_ecl[1], vz_ecl=v_ecl[2],
            x_earth=r_earth[0], y_earth=r_earth[1], z_earth=r_earth[2],
            vx_earth=v_ecl[0], vy_earth=v_ecl[1], vz_earth=v_ecl[2],
            vx_surface=v_surface[0], vy_surface=v_surface[1], vz_surface=v_surface[2],
            v_speed=v_speed,
            z_impact=z_impact,
        )
        # Extract elements as a dictionary using covlabel as keys
        elements = X_ecl[8:]
        elements_dict = {label: elements[j] for j, label in enumerate(covlabel)}
        location.update(elements_dict)
        locations += [location]

    locations = pd.DataFrame(locations)
    locations.sort_values('lon', inplace=True)

    ####################################################
    # Save impacts
    ####################################################
    print(f"Saving impacts to {file_impacts}")
    locations.to_csv(file_impacts, index=False)


Number of impacts: 30
Computing impact conditions on surface (lon, lat, time)
Saving impacts to data/objects/2024YR4/sim//impacts-jd0_246080050--enc_246358909.csv


## Map

In [32]:
import folium
import pandas as pd
import leafmap.foliumap as leafmap

In [33]:
# ################################################# #
# Results
# ################################################# #
candidates_file = f"{repo_dir}/candidates-{sim_suf}.csv"
candidates = pd.read_csv(candidates_file)
bodies_file = f"{repo_dir}/bodies-{sim_suf}.csv"
bodies = pd.read_csv(bodies_file)
impacts_file = f"{repo_dir}/impacts-{sim_suf}.csv"
impacts = pd.read_csv(impacts_file)

event_label = f"{obj_id} - Orbit ID: {orbit_id} ({orbit['orbit']['soln_date']})"

In [36]:
map = leafmap

center_lat = impacts['lat'].mean()
center_lon = impacts['lon'].mean()
center_lat,center_lon = 4,-74
impact_map = map.Map(location=[center_lat, center_lon], 
                        zoom_start=5,height=600)

line = folium.PolyLine(
    locations=impacts[['lat','lon']],
)
impact_map.add_child(line)
folium.LayerControl(collapsed=False).add_to(impact_map)

# Define the legend
"""
legend_dict = {
    "High": "red",
    "Medium": "yellow",
    "Low": "green"
}
impact_map.add_legend(title="Orbita",legend_dict=legend_dict)
"""

legend_html = f'''
<div style="position:fixed; top:10px; left:50px; height:1.5rem; z-index:9999; font-size:0.8em; background-color:white;margin:5px;">
    [PY]mpact Map / {orbit['spice_id']} / Orbit {orbit['orbit_id']} / Solution {orbit['orbit']['soln_date']} <br>
</div>
'''
legend_html = f'''
<div style="position:fixed; top:10px; left:50px; height:1.5rem; z-index:9999; font-size:0.8em; background-color:white;margin:5px;">
    [PY]mpact Map / {event_label} <br>
</div>
'''

impact_map.get_root().html.add_child(folium.Element(legend_html))
impact_map